# 04 — Entraînement Cloud (Google Colab GPU)
**NightVision Gen — Entraînement DDPM conditionnel depuis zéro (Modèles A, B, C)**

Ce notebook est conçu pour être exécuté sur **Google Colab avec accélérateur GPU**.
Il récupère le code depuis le dépôt GitHub, sauvegarde les checkpoints sur **Google Drive** de manière persistante, et permet de sélectionner la variante à entraîner sans toucher au code source.

---
### Plan d'exécution :
1. Vérification de l'environnement GPU (NVIDIA T4 / V100 / A100).
2. Montage de Google Drive pour la persistance des poids.
3. Clonage du dépôt GitHub et installation des dépendances.
4. Téléchargement ou accès aux données LOL-v2 et ExDark (décompression automatique).
5. Sélection de la variante : **Modèle A**, **Modèle B** ou **Modèle C**.
6. Configuration et redirection vers Google Drive.
7. Préparation des DataLoaders LOL-v2 avec patchs 128x128.
8. Lancement de l'entraînement avec le `Trainer` modulaire.
9. Visualisation des courbes de perte et des aperçus visuels.
10. Sauvegarde et synchronisation sur Google Drive.

In [ ]:
# 1. Vérification de l'environnement GPU
!nvidia-smi
import torch
print("PyTorch Version :", torch.__version__)
print("CUDA Disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Détecté     :", torch.cuda.get_device_name(0))
    print("Mémoire VRAM    :", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2), "Go")
else:
    print("⚠️ Aucun GPU détecté. Activez l'accélérateur GPU dans Exécution > Modifier le type d'exécution.")

In [ ]:
# 2. Montage de Google Drive — un seul dossier pour les datasets et les résultats
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')

def _is_project_folder(name: str) -> bool:
    key = name.lower().replace('_', '-').replace(' ', '')
    return key in {'nightvision-gen', 'nightvisiongen'}

def _contains_datasets(folder: Path) -> bool:
    markers = (
        'LOLv2', 'ExDark',
        'data/LOLv2', 'data/ExDark',
        'data/raw/LOLv2', 'data/raw/ExDark',
    )
    return any((folder / rel).exists() for rel in markers)

candidates = []
if DRIVE_ROOT.exists():
    candidates = [p for p in DRIVE_ROOT.iterdir() if p.is_dir() and _is_project_folder(p.name)]

with_data = [p for p in candidates if _contains_datasets(p)]
pool = with_data or candidates
if pool:
    # Préfère le dossier qui contient déjà LOLv2 / ExDark, puis le nom exact nightvision-gen.
    GDRIVE_DIR = sorted(pool, key=lambda p: (p.name != 'nightvision-gen', len(p.name)))[0]
else:
    GDRIVE_DIR = DRIVE_ROOT / 'nightvision-gen'
    GDRIVE_DIR.mkdir(parents=True, exist_ok=True)

duplicates = [p for p in candidates if p.resolve() != GDRIVE_DIR.resolve()]
print('✅ Dossier unique du projet :', GDRIVE_DIR)
print('   Datasets, checkpoints, logs et images générées restent dans ce dossier.')
if duplicates:
    print('⚠️ Dossiers en double ignorés. Déplacez leur contenu dans le dossier unique, puis supprimez-les :')
    for dup in duplicates:
        print('   -', dup)

In [ ]:
# 3. Clone du dépôt GitHub Public et installation du package
import os, sys
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/HASSANDADDA-04/nightvision-gen.git"

%cd /content

repo_dir = Path("/content/nightvision-gen")
if not repo_dir.exists():
    print(f"📥 Clonage du dépôt public {GITHUB_REPO_URL}...")
    !git clone $GITHUB_REPO_URL
else:
    print("🔄 Mise à jour du dépôt avec les derniers commits...")
    %cd /content/nightvision-gen
    !git pull
    %cd /content

%cd /content/nightvision-gen

# Installation des dépendances et installation physique du package
print("📦 Installation des dépendances Colab...")
!pip install -r requirements-colab.txt
print("📦 Installation standard du package nightvision...")
!pip install . --no-build-isolation --no-deps

# Nettoyage et invalidation des caches d'import Python
import importlib
sys.modules.pop("nightvision", None)
importlib.invalidate_caches()

# Enregistrement explicite dans sys.path pour un accès direct immédiat
src_dir = str(Path("/content/nightvision-gen/src").resolve())
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

try:
    import nightvision
    print("✅ Package nightvision installé et importé avec succès sur Colab !")
except ModuleNotFoundError:
    print("
⚠️ Note Colab : Pour finaliser l'enregistrement dans Python 3.13 :")
    print("👉 Allez dans le menu : Exécution > Redémarrer la session (Runtime > Restart session)")
    print("👉 Puis relancez cette cellule : l'import sera immédiat !")

In [ ]:
# 4. Accès aux datasets déjà présents dans le dossier unique du Drive
import sys, zipfile, shutil
from pathlib import Path
sys.path.append(str(Path("/content/nightvision-gen/src").resolve()))

Path("artifacts/checkpoints").mkdir(parents=True, exist_ok=True)
Path("artifacts/logs").mkdir(parents=True, exist_ok=True)
Path("artifacts/generated_images").mkdir(parents=True, exist_ok=True)
Path("data/raw/LOLv2").mkdir(parents=True, exist_ok=True)
Path("data/raw/ExDark").mkdir(parents=True, exist_ok=True)
Path("data/processed").mkdir(parents=True, exist_ok=True)

def _find_root(base: Path, folder_name: str, markers: tuple[str, ...]) -> Path | None:
    probes = [
        base / "data" / "raw" / folder_name,
        base / "data" / folder_name,
        base / "raw" / folder_name,
        base / folder_name,
        base / "datasets" / folder_name,
    ]
    seen = []
    for probe in probes:
        seen.extend((probe, probe / folder_name))
    for cand in seen:
        if any((cand / marker).is_dir() for marker in markers):
            return cand
    return None

def _find_zip(base: Path, needle: str) -> Path | None:
    skip = {"LOLv2", "ExDark", "checkpoints", "logs", "generated_images", "data", "artifacts"}
    levels = [base]
    for depth in range(3):
        nxt = []
        for folder in levels:
            try:
                children = list(folder.iterdir())
            except OSError:
                continue
            for child in children:
                if child.is_file() and child.suffix.lower() == ".zip" and needle in child.name.lower():
                    return child
                if child.is_dir() and depth < 2 and child.name not in skip:
                    nxt.append(child)
        levels = nxt
    return None

def _copy_tree(src: Path, dest: Path) -> None:
    dest.mkdir(parents=True, exist_ok=True)
    for item in src.iterdir():
        target = dest / item.name
        if item.is_dir():
            shutil.copytree(item, target, dirs_exist_ok=True)
        elif not target.exists():
            shutil.copy2(item, target)

def _extract_zip(zip_path: Path, dest_name: str) -> None:
    print(f"📦 Décompression de {zip_path.name}...")
    dest = Path("data/raw") / dest_name
    with zipfile.ZipFile(zip_path, "r") as z:
        first_entry = z.namelist()[0].strip("/")
        if first_entry == dest_name or first_entry.startswith(dest_name + "/"):
            z.extractall("data/raw")
        else:
            z.extractall(dest)
    nested = dest / dest_name
    if nested.exists():
        for item in nested.iterdir():
            target = dest / item.name
            if not target.exists():
                shutil.move(str(item), str(target))
        try:
            nested.rmdir()
        except OSError:
            pass

# LOL-v2 : lu dans le dossier Drive unique, copié seulement sur le SSD Colab (session temporaire).
lol_src = _find_root(GDRIVE_DIR, "LOLv2", ("Real_captured/Train/Low", "Real_captured", "Low"))
lol_local = Path("data/raw/LOLv2")
lol_ready_marker = lol_local / "Real_captured" / "Train" / "Low"
if not lol_ready_marker.exists() and not (lol_local / "Low").exists():
    if lol_src is not None:
        print("📂 LOL-v2 trouvé dans", lol_src)
        print("   Copie temporaire vers le SSD Colab pour l'entraînement (le dossier Drive n'est pas dupliqué).")
        _copy_tree(lol_src, lol_local)
    else:
        lol_zip = _find_zip(GDRIVE_DIR, "lol")
        if lol_zip is not None:
            _extract_zip(lol_zip, "LOLv2")

# ExDark : même dossier Drive.
exdark_src = _find_root(GDRIVE_DIR, "ExDark", ("Bicycle", "Car", "People", "Images"))
exdark_local = Path("data/raw/ExDark")
if not (exdark_local / "Bicycle").exists() and not (exdark_local / "Car").exists():
    if exdark_src is not None:
        print("📂 ExDark trouvé dans", exdark_src)
        _copy_tree(exdark_src, exdark_local)
    else:
        exdark_zip = _find_zip(GDRIVE_DIR, "exdark")
        if exdark_zip is not None:
            _extract_zip(exdark_zip, "ExDark")

lol_ready = (lol_local / "Real_captured" / "Train" / "Low").exists() or (lol_local / "Low").exists()
exdark_ready = (exdark_local / "Bicycle").exists() or (exdark_local / "Car").exists()
print("Statut LOL-v2 :", "✅ Détecté et prêt" if lol_ready else "⚠️ Non détecté dans " + str(GDRIVE_DIR))
print("Statut ExDark :", "✅ Détecté et prêt" if exdark_ready else "⚠️ Non détecté dans " + str(GDRIVE_DIR))
if not lol_ready:
    print("Structure attendue dans le dossier unique :")
    print("  nightvision-gen/LOLv2/Real_captured/Train/Low")
    print("  nightvision-gen/ExDark/Bicycle")
    print("  nightvision-gen/checkpoints  |  logs  |  generated_images")

In [ ]:
# 5. Sélection de la variante du modèle à entraîner
# @title Configuration de l'Expérience { run: "auto" }
MODEL_VARIANT = "A" # @param ["A", "B", "C"]
EPOCHS = 100 # @param {type:"integer"}
BATCH_SIZE = 16 # @param {type:"integer"}
LEARNING_RATE = 1e-4 # @param {type:"number"}
LAMBDA_EDGE = 0.05 # @param {type:"number"}
RESUME_TRAINING = False # @param {type:"boolean"}

print(f"🎯 Variante sélectionnée : Modèle {MODEL_VARIANT}")
print(f"   Époques : {EPOCHS} | Batch size : {BATCH_SIZE} | LR : {LEARNING_RATE}")
if MODEL_VARIANT == "C":
    print(f"   Poids de la perte de contours (lambda_edge) : {LAMBDA_EDGE}")

In [ ]:
# 6. Chargement de la configuration YAML correspondante
from nightvision.utils.config import load_config

config_file = f"configs/train_{MODEL_VARIANT}.yaml"
cfg = load_config(config_file)

# Mise à jour avec les paramètres du notebook
cfg["training"]["epochs"] = EPOCHS
cfg["training"]["batch_size"] = BATCH_SIZE
cfg["training"]["learning_rate"] = LEARNING_RATE
if MODEL_VARIANT == "C":
    cfg["loss"]["lambda_edge"] = LAMBDA_EDGE

# Redirection de la sauvegarde vers Google Drive pour éviter toute perte en cas de déconnexion
drive_ckpt_dir = GDRIVE_DIR / f"checkpoints/model_{MODEL_VARIANT}"
drive_ckpt_dir.mkdir(parents=True, exist_ok=True)
cfg["paths"]["checkpoint_dir"] = str(drive_ckpt_dir)

drive_log_dir = GDRIVE_DIR / f"logs/model_{MODEL_VARIANT}"
drive_log_dir.mkdir(parents=True, exist_ok=True)
cfg["paths"]["log_dir"] = str(drive_log_dir)

drive_gen_dir = GDRIVE_DIR / f"generated_images/model_{MODEL_VARIANT}"
drive_gen_dir.mkdir(parents=True, exist_ok=True)
cfg["paths"]["generated_dir"] = str(drive_gen_dir)

print("Configuration prête.")
print("  - Checkpoints :", drive_ckpt_dir)
print("  - Logs CSV    :", drive_log_dir)
print("  - Images gen  :", drive_gen_dir)

In [ ]:
# 7. Préparation des DataLoaders LOL-v2
from torch.utils.data import DataLoader
from nightvision.data.split_manager import LOLv2SplitManager
from nightvision.data.dataset import LOLv2PatchDataset, SyntheticPairDataset

compute_retinex = MODEL_VARIANT in ("B", "C")
manifest_path = Path("data/processed/lol_v2_splits.json")

train_pairs = []
val_pairs = []

if manifest_path.exists():
    print("Lecture du manifeste :", manifest_path)
    splits = LOLv2SplitManager.load_split_manifest(manifest_path, root_dir="data/raw/LOLv2")
    train_pairs = splits.get("train", [])
    val_pairs = splits.get("val", [])

# Vérifier si les chemins pointent vers des fichiers réels existants
if not train_pairs or not Path(train_pairs[0].low_path).exists():
    print("Découverte dynamique des paires LOL-v2 depuis data/raw/LOLv2...")
    lol_mgr = LOLv2SplitManager(root_dir="data/raw/LOLv2", seed=42)
    splits = lol_mgr.create_splits()
    train_pairs = splits.get("train", [])
    val_pairs = splits.get("val", [])

if len(train_pairs) > 0 and Path(train_pairs[0].low_path).exists():
    print(f"✅ Dataset réel LOL-v2 opérationnel : {len(train_pairs)} paires train, {len(val_pairs)} paires val.")
    train_ds = LOLv2PatchDataset(train_pairs, patch_size=128, patches_per_image=4, is_train=True, compute_retinex=compute_retinex)
    val_ds = LOLv2PatchDataset(val_pairs, patch_size=128, patches_per_image=2, is_train=False, compute_retinex=compute_retinex)
else:
    print("⚠️ Images LOL-v2 réelles non disponibles. Utilisation du dataset synthétique.")
    train_ds = SyntheticPairDataset(32, image_size=128, compute_retinex=compute_retinex)
    val_ds = SyntheticPairDataset(8, image_size=128, compute_retinex=compute_retinex)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print(f"DataLoaders prêts : {len(train_loader)} batches par époque.")

In [ ]:
# 8. Initialisation du Trainer et Lancement de l'Entraînement
from nightvision.training.trainer import Trainer

trainer = Trainer(config=cfg, train_loader=train_loader, val_loader=val_loader)

# Reprise éventuelle depuis un checkpoint existant
latest_ckpt = drive_ckpt_dir / "checkpoint_latest.pt"
if RESUME_TRAINING and latest_ckpt.exists():
    print("🔄 Reprise de l'entraînement depuis :", latest_ckpt)
    trainer.resume_from_checkpoint(latest_ckpt)

# Lancement de l'entraînement
print(f"🚀 Démarrage de l'entraînement du Modèle {MODEL_VARIANT} depuis zéro...")
trainer.train()

In [ ]:
# 9. Visualisation des Courbes d'Apprentissage et du Dernier Aperçu
import json
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

log_dir_path = Path(cfg["paths"]["log_dir"])

# Chargement robuste : tentative prioritaire via JSON (structure parfaite), repli sur CSV sécurisé
json_files = list(log_dir_path.glob("*_history.json")) + list(Path("artifacts/logs").rglob("*_history.json"))
csv_files = list(log_dir_path.glob("*_metrics.csv")) + list(Path("artifacts/logs").rglob("*_metrics.csv"))

df = None
if json_files and json_files[0].exists():
    try:
        with open(json_files[0], "r", encoding="utf-8") as f:
            data = json.load(f)
        df = pd.DataFrame(data)
    except Exception:
        pass

if df is None and csv_files and csv_files[0].exists():
    try:
        df = pd.read_csv(csv_files[0], on_bad_lines="skip")
    except Exception as e:
        print("Erreur lecture CSV :", e)

if df is not None and not df.empty:
    plt.figure(figsize=(10, 4))
    if "epoch" in df.columns and "train_loss" in df.columns:
        plt.plot(df["epoch"], df["train_loss"], label="Perte Entraînement (Train)", color="#3b82f6", lw=2)
    if "epoch" in df.columns and "val_loss" in df.columns:
        val_df = df.dropna(subset=["val_loss"])
        if not val_df.empty:
            plt.plot(val_df["epoch"], val_df["val_loss"], label="Perte Validation (Val)", color="#ef4444", lw=2, marker="o")
    plt.xlabel("Époque")
    plt.ylabel("Perte")
    plt.title(f"Historique d'Entraînement — Modèle {MODEL_VARIANT}")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.7)
    plt.tight_layout()
    plt.show()
else:
    print("Fichier de métriques en attente de génération.")

# Aperçu de la dernière validation
gen_dir_path = Path(cfg["paths"]["generated_dir"])
preview_imgs = sorted(list(gen_dir_path.glob("val_epoch_*.png")) + list(Path("artifacts/generated_images").rglob("val_epoch_*.png")))
if preview_imgs:
    last_preview = preview_imgs[-1]
    print(f"🖼️ Dernier aperçu visuel généré : {last_preview.name}")
    plt.figure(figsize=(12, 4))
    plt.imshow(Image.open(last_preview))
    plt.axis("off")
    plt.title(f"Aperçu Restauration — {last_preview.stem}")
    plt.tight_layout()
    plt.show()

In [ ]:
# 10. Synchronisation finale sur Google Drive
!cp -r artifacts/logs/* {str(GDRIVE_DIR)}/logs/ 2>/dev/null || true
!cp -r artifacts/generated_images/* {str(GDRIVE_DIR)}/generated_images/ 2>/dev/null || true
print("✅ Checkpoints, logs et aperçus synchronisés sur Google Drive :", GDRIVE_DIR)